In [1]:
!pip install -qU langchain_community pypdf langchain_huggingface
!pip install -qU sentence-transformers faiss-cpu langchain langchain-groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.7/309.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [2]:
import torch
import faiss
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

load_dotenv()

True

In [ ]:
def load_all_documents(folder_path: str):
    all_docs = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            docs = loader.load()
            all_docs.extend(docs)
    return all_docs


# 3. Chunking
def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
    return splitter.split_documents(docs)

all_docs = load_all_documents("./docs")

chunks = split_documents(all_docs)

In [4]:
# print(len(all_docs))
# print(len(chunks))

377
1973


In [5]:
# 4. Embeddings
def build_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    model_kwargs = {'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
    encode_kwargs = {'normalize_embeddings': True}
    return HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)


embedding = build_embeddings()

vectordb = FAISS.from_documents(chunks, embedding)
# Save the FAISS index to disk
save_path = "faiss_index"
vectordb.save_local(save_path)
print(f"FAISS index saved to {save_path}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index saved to faiss_index


In [6]:
# print(vectordb)

query = input("Enter your query: ")
retrieved_docs = vectordb.similarity_search(query, k=5)

context = "\n\n\n".join([doc.page_content for doc in retrieved_docs])
print("Context:\n", context)

Enter your query: what is disease
Context:
 is very variable; some individuals dominate or manipulate family
and social networks as a result of their symptoms, in contrast to
a minority who function almost normally.
Diagnostic guidelines
For a definite diagnosis, both of the following should be present:
(a) persistent belief in the presence of at least one serious physical
illness underlying the presenting symptom or symptoms, even


F00-F09 ORGANIC MENTAL DISORDERS
systemic disease or disorder. The term "symptomatic" is used for those organic
mental disorders in which cerebral involvement is secondary to a systemic
extracerebral disease or disorder.
It follows from the foregoing that, in the majority of cases, the recording
of a diagnosis of any one of the disorders in this block will require the use


the criteria that govern the diagnosis of more specific types.
Dementia is a syndrome due to disease of the brain, usually of a chronic or
progressive nature, in which there is disturba

In [7]:
prompt_template = """
You are a helpful assistant. Answer the question based only on the provided context from documents.
If the answer is not in the context, respond with "The answer is not available in the provided documents."

Include the page number from the context as a reference in your answer, like this: (Page X).

Context:
{context}

Question:
{question}

Answer:
"""


# Load into Langchain prompt
QA_PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

In [8]:
# Create LLM
llm = ChatGroq(
    temperature=0.25,
    model_name="meta-llama/llama-4-scout-17b-16e-instruct",
    model_kwargs={"max_completion_tokens": 1024}
)

# Build RAG pipeline with custom prompt
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(),
    chain_type="stuff",
    chain_type_kwargs={"prompt": QA_PROMPT}
)

In [ ]:
query = input("Ask your question: ")
response = qa_chain.invoke({"query":query})
response

Ask your question: what is demenia?


{'query': 'what is demenia?',
 'result': 'Dementia is a condition characterized by a decline in cognitive abilities, and although this decline is essential for its diagnosis, no consequent interference with the performance of social roles is used as a diagnostic guideline (Page 335).'}

In [ ]:
print(response['result'])

Dementia is a condition characterized by a decline in cognitive abilities, and although this decline is essential for its diagnosis, no consequent interference with the performance of social roles is used as a diagnostic guideline (Page 335).
